# 04 - Pipelines Avanzados con `ColumnTransformer`, `passthrough` y Benchmark Global

## Curso de Machine Learning: Modelos de Clasificación

En proyectos reales de Ingeniería de Machine Learning, cada modelo dentro de un metamodelo puede requerir un preprocesamiento distinto (por ejemplo, escalar variables para `SVC`/`KNN`, pero pasar los datos crudos a `RandomForest` o `HistGradientBoosting`), o bien habilitar `passthrough=True` en `StackingClassifier` para que el Meta-Clasificador observe tanto las predicciones de Nivel 0 como las variables originales.

---

### Contenidos
1. Efecto de `passthrough=True` vs `passthrough=False` en `StackingClassifier`
2. Diseño de Pipelines Especializados por Subconjunto de Columnas (`ColumnTransformer`)
3. Benchmark Global Definitivo: Modelos Individuales vs Ensambles Homogéneos vs Metamodelos (Voting y Stacking)


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Comparacion de Stacking con passthrough=False vs passthrough=True
cancer = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target)

base_estimators = [
    ('lr', make_pipeline(StandardScaler(), LogisticRegression(random_state=42))),
    ('rf', RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1)),
    ('svc', make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))),
    ('hgb', HistGradientBoostingClassifier(random_state=42))
]

stack_standard = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    passthrough=False, cv=5, n_jobs=-1
).fit(X_tr, y_tr)

# Con passthrough=True, escalamos en el final_estimator porque recibe tambien X original
stack_passthrough = StackingClassifier(
    estimators=base_estimators,
    final_estimator=make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
    passthrough=True, cv=5, n_jobs=-1
).fit(X_tr, y_tr)

print(f"Stacking (passthrough=False) -> Dimension entrada Meta-Modelo: {stack_standard.final_estimator_.coef_.shape[1]} columnas | Test Acc: {stack_standard.score(X_te, y_te)*100:.2f}%")
print(f"Stacking (passthrough=True)  -> Dimension entrada Meta-Modelo: {stack_passthrough.final_estimator_[-1].coef_.shape[1]} columnas | Test Acc: {stack_passthrough.score(X_te, y_te)*100:.2f}%")


In [ ]:
# Benchmark Global de todos los paradigmas de Clasificacion
catalogo_global = {
    '1. Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
    '2. Gaussian Naive Bayes': GaussianNB(),
    '3. KNN (K=7)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    '4. SVC (RBF Kernel)': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42)),
    '5. Random Forest': RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1),
    '6. AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42),
    '7. HistGradientBoosting': HistGradientBoostingClassifier(random_state=42),
    '8. Voting Classifier (Soft)': VotingClassifier(estimators=base_estimators, voting='soft', n_jobs=-1),
    '9. Stacking Classifier': stack_standard
}

resultados_globales = []
for nombre, clf in catalogo_global.items():
    t0 = time.perf_counter()
    clf.fit(X_tr, y_tr)
    t_ms = (time.perf_counter() - t0) * 1000
    
    preds = clf.predict(X_te)
    probs = clf.predict_proba(X_te)[:, 1]
    resultados_globales.append({
        'Modelo': nombre,
        'Accuracy (%)': accuracy_score(y_te, preds) * 100,
        'F1-Score (%)': f1_score(y_te, preds) * 100,
        'ROC-AUC (%)': roc_auc_score(y_te, probs) * 100,
        'Log-Loss': log_loss(y_te, probs),
        'Tiempo Fit (ms)': t_ms
    })

df_global = pd.DataFrame(resultados_globales).sort_values(by='Log-Loss', ascending=True)
print(df_global.to_string(index=False))
